# BioLab Remote MedGemma Server on Colab

This notebook runs only the MedGemma model service on Colab GPU. Your local PC continues to run the BioLab app, Kafka, PySpark outputs, AI backend, and SQLite database.

In [ ]:
!nvidia-smi

Clone the repository. Skip this cell if you uploaded the repo manually.

In [ ]:
!git clone https://github.com/aminebouanani/BioLab-Project.git
%cd BioLab-Project

In [ ]:
!pip install -r medgemma_server/requirements.txt

Authenticate to Hugging Face. Prefer setting `HF_TOKEN` as an environment variable. If needed, run `!huggingface-cli login`.

In [ ]:
import os
# os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN'
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import os, subprocess, time
os.environ['MEDGEMMA_MODEL_ID'] = os.environ.get('MEDGEMMA_MODEL_ID', 'google/medgemma-4b-it')
os.environ['MEDGEMMA_MAX_NEW_TOKENS'] = os.environ.get('MEDGEMMA_MAX_NEW_TOKENS', '512')
os.environ['MEDGEMMA_TEMPERATURE'] = os.environ.get('MEDGEMMA_TEMPERATURE', '0.2')
server = subprocess.Popen(['uvicorn', 'medgemma_server.app:app', '--host', '0.0.0.0', '--port', '9000'])
time.sleep(5)
print('Server started on port 9000')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess, re, time
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:9000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(60):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[-a-zA-Z0-9.]+trycloudflare.com', line)
    if match:
        public_url = match.group(0)
        break
print('PUBLIC_URL=', public_url)

In [ ]:
import requests
print(requests.get(public_url + '/health').json())

In [ ]:
import requests, json
payload = {
  'context': {
    'patient_id': 'PAT-sample', 'order_id': 'ORD-sample', 'specimen_id': 'SPC-sample',
    'results_count': 1, 'abnormal_results_count': 1, 'normal_results_count': 0,
    'unknown_flag_results_count': 0, 'validation_status_summary': 'FINAL',
    'context_hash': 'sample-context-hash',
    'results': [{'test_name': 'Glucose', 'loinc_code': '2345-7', 'value_raw': '160', 'unit': 'mg/dL', 'reference_range': '70-110', 'abnormal_flag': 'H', 'validation_status': 'FINAL', 'result_datetime': '2026-01-01T10:00:00Z'}]
  },
  'instructions': 'Generate a structured AI draft biological report using only the provided context.'
}
print(requests.post(public_url + '/generate-report', json=payload, timeout=180).json())

Run these commands locally in PowerShell, replacing `<public_url>` with the printed URL:

```powershell
$env:AI_PROVIDER="remote_medgemma"
$env:AI_PROVIDER_FALLBACK_TO_MOCK="false"
$env:REQUIRE_REAL_LLM="true"
$env:MEDGEMMA_API_URL="<public_url>"
uvicorn ai_backend.app.main:app --reload --port 8001
```

Then test:

```powershell
python scripts/test_remote_medgemma_provider.py
python scripts/test_full_app_real_llm.py
```

Colab runs only the MedGemma model service. Later, the same `MEDGEMMA_API_URL` can point to Azure ML, Azure AI Foundry, or an Azure GPU VM.